# 토큰과 토그나이징, 토크나이저

*코퍼스*란?
> 코퍼스는 겹겹이 쌓인 구조를 가지고 있다.
>Corpus (말뭉치): 전체 데이터셋 (예: 네이버 영화 리뷰 20만 개 전체)

>Document (문서): 코퍼스를 구성하는 개별 단위 (예: 리뷰 1건, 뉴스 기사 1개)

>Sentence (문장): 문서 내의 논리적 단위 (예: "이 영화 정말 재밌네요.")

>Token (토큰): 더 이상 쪼갤 수 없는 분석의 최소 단위 (예: "이", "영화", "정말", "재밌", "네요")

*토큰화란?*

텍스트를 모델이 처리할 수 있는 작은 단위 즉, 토큰으로 쪼개는 과정이다. 

토큰화 방식들

단어기반(word based): '학교는'과 '학교가' 가 다르게 토큰화된다 -> 비효율  
문자기반(char based): 의미가 파괴된다.  
형태소 기반: 의미 최소 단위인 형태소로 토큰화한다. 하지만, OOV 문제가 있다.

**subword based**: 서브워드 방식은 요즘 대세! LLM이 사용하는 방식이다.

## 전통적 NLP에서 **언어 전처리 과정**

텍스트 -> 토큰화 -> 정제(html, 특수문자 제거) -> 형태소, 표제어 추출(영어: Stemming, Lemmatization) -> 불용어 제거(의미 없는 단어 삭제. 예를 들어 i my me) -> 정규표현식 사용 -> 인코딩 -> 패딩(인코딩된 값 문장 길이 맞추기)


전통적인 NLP -> Feature Engineering 중심이다.
> 딥러닝 시대 이전 전통적 NLP에는 모델 자체가 똑똑하지 않았기 때문에 사람이 얼마나 공들여 데이터를 정제하는지가 즉, Feature Engineering이 모델의 성능을 결정했다.  
> 하지만, 요즘은 BERT나 GPT 등 모델은 NLP의 전처리 과정을 **인간이 닦고 조이는 방식에서 모델이 알아서 배우는 방식으로 완전히 뒤바꿨다**

## LLM 시대의 파이프라인 - 극적인 단순화!

요즘은 각 모델마다 각각의 토크나이저 vocab를 가지고 등장한다. 텍스트를 입력하면 그 모델의 토크나이저 사전으로 분류할 수 있는 것이다.
> 즉, 이전에는 일일히 수작업 하던 것을 이제는 잘 만들어진 토크나이저 하나가 알아서 한다!  

텍스트 -> 토크나이저 -> Token IDs -> 모델입력

Token IDs란?
> 텍스트를 모델이 처리할 수 있는 숫자로 변환했을 때, 각 토큰에 부여된 값이다. 즉, 인공지능의 단어 사전 인덱스 번호이다. 
> 토크나이저로 문장을 쪼갬 -> 모델이 미리 공부해서 가지고 있는 vocab(어휘사전)에서 해당 조각이 몇 번째에 있는지 찾기 -> Token IDs로 변환함 -> 이 ID는 임베딩의 좌표가 된다(임베딩 행렬에서 어떤 줄을 읽어올지 결정하는 주소가 바로 이 Token IDs이다!)


## 각 모델마다 토크나이저가 있다고?

학습과정: 코퍼스 수집 -> 알고리즘(BPE ,Word Piece)과 사전크기 |V| 결정 -> 학습

이렇게 학습이 되면, 토크나이저는 **vocab와 Merge rule(병합규칙)**을 가져온다.  
vocab는 토큰과 ID를 매칭한 것이고 병합규칙은 어떤 순서로 글자를 합칠지 적어둔 리스트이다. 
> 이 두가지가 합쳐질 때 우리는 그것을 그 모델이 학습한 토크나이저라고 부른다. (예를 들어 BPE 토크나이저는 그것의 vocab와 병합규칙을 가지고 있을 것이다)

새로운 문장이 들어왔을 때 토크나이저는 문장을 글자 단위로 쪼갠 후 병합규칙에 따라 조합한다 (l/o/v/e -> 규칙 확인 -> love) 그리고 그 결과로 Token IDs를 뱉어낸다!\


각 모델마다 각자의 토크나이저(그리고 vocab)가 있기 때문에 같은 문장도 모델마다 다르게 쪼갠다. 여기서 성능 차이가 발생한다. 
> 한국어를 잘 토큰화하는 모델은 그들이 학습한 vocab에 한국어가 많은 것이다. 

subwords 토크나이징 방법론에는 BPE, Word Piece, SentencePiece가 있다. 

## BPE (Subword 토크나이징 방법론. 대체로!)

BPE는 이전 토크나이저 알고리즘 중 하나이다. 

1. 모든 단어를 글자로 쪼갬  
2. (코퍼스 내에서) 빈도 높은 쌍을 병합.  
3. 병합한 쌍을 새로운 단어로 하고 반복 수행  
4. 최종 vocab 완성 

이렇게 하면 OOV 문제 해결!
> lowest라는 신조어가 들어옴 -> 일단 글자단위로 토크나이저가 다 쪼갬 -> 학습한 병합규칙 적용해서 low와 est를 만듦 -> ID 변환함 - > 최종출력이 두 ID인 [450, 820]이라고 한다면, 모델은 임베딩 할 때 450번 벡터와 820번 벡터를 두 개 꺼내와서 low 벡터를 통해 낮다의 의미를 est 벡터를 통해 최상급이라는 의미를 파악할 것이다.  
> 결국 임베딩을 하면 이런 신조어의 경우 두개의 행벡터를 차곡차곡 쌓아 (2,d ) 행렬로 처리하겠지. 

## vocabulary가 중요한 이유

1. 모델은 숫자만 본다. 모델이 텍스트를 받으면 그것은 토크나이저가 가지고 있는 Token ID의 시퀀스일 뿐이다
2. 같은 문장도 토크나이저 + (그것이 학습한) vocab에 따라 다르게 쪼갠다 (토크나이저가 학습한 결과물이 vocab)
3. 토큰 분할 방식 자체가 모델 성능에 직결된다. 

## SBERT

Sentence BERT는 패키지이다!! 토큰화, 임베딩까지 한번에 해결해준다!!

우리는 문장만 넣어주면? -> 모델의 전용 토크나이저가 문장을 쪼개서 Token IDs를 만들고, 이 ID들이 모델 본체로 들어가 임베딩 된다.

> 즉 한 방 패키지!
>

근데 S BERT는 여러 개의 토큰을 하나의 문장 벡터로 만들기를 원한다. 즉, 문장 전체를 대표하는 하나의 숫자를 원한다

그래서 mean Pooling 방식 등 으로 모든 토큰의 벡터 값을 평균낸다.

S BERT를 쓰면, 문장 간 유사도를 구할 때 문장을 대표하는 벡터끼리의 코사인 유사도만 구하면 된다. 또한, S BERT를 통해 벡터화해서 좌표평면에 뿌리면? 군집화도 가능함. 언급한 대로 전통적 방식처럼 불용어 지우고 전처리 하는 과정 필요가 없음!

> 군집화 알고리즘(예: K-Means)을 돌리려면 데이터가 좌표평면 위의 **'점'**으로 표현되어야 합니다. SBERT는 어떤 길이의 문장이 들어와도 768차원(혹은 모델에 따라 다른 고정 차원)의 벡터 하나로 딱 뱉어주기 때문에, 바로 군집화 알고리즘에 집어넣을 수 있습니다.

**SBERT를 활용한 군집화 파이프라인**  


임베딩 (Embedding): SBERT를 사용해 모든 문장을 벡터로 바꾼다. 

차원 축소 (Dimensionality Reduction): 768차원은 너무 복잡해서 군집화가 잘 안 될 수 있습니다. UMAP이나 t-SNE 같은 알고리즘으로 2~5차원 정도로 줄인다.

군집화 (Clustering): K-Means나 HDBSCAN 같은 알고리즘을 사용해 비슷한 벡터들끼리 그룹을 묶는다. 

분석 (Analysis): 각 군집에 어떤 단어들이 많이 들어있는지 확인하여 "이 그룹은 정치 기사구나", "이 그룹은 연예 기사구나"라고 해석한다. 
